In [ ]:
import os
from google.oauth2 import service_account
from googleapiclient.discovery import build
from openai import OpenAI

# Constants
SPREADSHEET_ID = '1wDaFAe5ayIB8zSyjub9QQfsFtyQcHdbgOR4YZmeY8vQ'  
SHEET_NAME = 'Sheet2'
CREDENTIALS_FILE = 'data/url-to-email-445616-cebe4868914f.json'  
OPENAI_API_KEY = ""
# Category ranking
category_ranking = {
	"TESTIMONIALS": 1,
	"COURSES": 2,
	"SERVICES": 3,
	"WEBINAR": 4,
	"PODCAST": 5,
	"EBOOK": 6,
	"RECENT_BLOG": 7,
	"ABOUT_US": 8,
	"SHOP": 9
}

# Column indices for categories (0-based, A=0, B=1, ..., M=12, N=13, ..., U=20)
category_indices = {
	"ABOUT_US": 12,  # M
	"EBOOK": 13,     # N
	"COURSES": 14,   # O
	"RECENT_BLOG": 15, # P
	"TESTIMONIALS": 16, # Q
	"WEBINAR": 17,   # R
	"SERVICES": 18,  # S
	"PODCAST": 19,   # T
	"SHOP": 20       # U (excluded)
}

# Output column letters
output_columns = {
	"Email 1": "W",
	"Email 1 Data Point": "X",
	"Subsequence 1": "Y",
	"Subsequence 1 Data Point": "Z",
	"Subsequence 2": "AA",
	"Subsequence 2 Data Point": "AB",
	"Subsequence 3": "AC",
	"Subsequence 3 Data Point": "AD",
	"Subsequence 4": "AE",
	"Subsequence 4 Data Point": "AF",
	"Email 2": "AG",
	"Email 3": "AH"
}

# Prompt templates
prompt_templates = {
"Email 1": {
	"TESTIMONIALS": """
		Example email - 

		James - Saw your work with Kyle and how you helped him 10x his revenue — love that!

		I know that a ton of folks are struggling to make the same leap.

		Assuming, if we could 10X your exposure in 45 days without you lifting a finger…

		We’ll do this by using the insights from your existing stuff and feature them on our network of High Traffic Pages.

		It will build authority and make strangers flock to your paid programs.

		One client saw a 287% Increase in new buyers almost instantly.

		Would you like to know how?


		Template email - 

		[FIRST NAME] - Saw your work [PERSONALISATION - Describe the result/case study/testimonial/review based on the data point and how it helped their prospect] — love that!

		I know that a ton of folks are struggling to make the same leap.

		Assuming, if we could 10X your exposure in 45 days without you lifting a finger…

		We’ll do this by using the insights from your existing stuff and feature them on our network of High Traffic Pages.

		It will build authority and make strangers flock to your paid programs.

		One client saw a 287% Increase in new buyers almost instantly.

		Would you like to know how?
	""",
	"COURSES": """
		Example email - 

		James - Just looked in your Real Estate Moguls course. 

		Sooo many people could use that!

		Assuming, If we could take the insights & frameworks from it and feature them on our network of High Traffic Pages.

		One client saw a 287% Increase in online course purchases. 

		There’s so much value here, people would go from strangers to flocking to the course real fast.

		Would you like to know more?

		Template email - 

		[First Name] - Just looked into your [PERSONALISATION - Name of the course/program course/program (Whether the suffix is a course or program will depend on what the data point is)]. 

		Sooo many people could use that!

		Assuming, If we could take the insights & frameworks from it and feature them on our network of High Traffic Pages.

		One client saw a 287% Increase in online course purchases. 

		There’s so much value here, people would go from strangers to flocking to the [course/program (Like mentioned above, whether it is a course or program will depend on what the data point is)] real fast.

		Would you like to know more?
	""",
	"SERVICES": """
		Example email - 

		James - Just checked out your Construction Technology Services page . 

		Sooo many Contractors could use that!

		Assuming if we could use our network of High Traffic Pages to get you more Contractors 

		There’s so much value here, people would turn from strangers to customers real fast.

		Our client Katie saw a 287% jump in new clients by doing this. 

		Would you like to know more?

		Template email - 

		[FIRST NAME] - Just checked out your [PERSONALISATION - Name of the Service mentioned on the Services/Consultaing Page Depending on what it is based on the Data Point] page . 

		Sooo many [PERSONALISATION - Mention the most probable ICP based on the data point that could use that particular type of service/consulting] could use that!

		Assuming if we could use our network of High Traffic Pages to get you more [PERSONALISATION - Same as the above – Mention the most probable ICP based on the data point that could use that particular type of service/consulting]

		There’s so much value here, people would turn from strangers to customers real fast.

		Our client Katie saw a 287% jump in new clients by doing this. 

		Would you like to know more?
	""",
	"WEBINAR": """
		Example email - 

		James - Just came across your Client ascension webinar — the focus on landing clients without actually going on the calls yourself sounds like a game-changer!

		This kind of value d eserves a much bigger audience.
    
		Assuming we could 10X your exposure in 45 days, without you lifting a finger…
    
		We’ll do this by using the insights from the webinar and featuring them on our network of High Traffic Pages.
    
		It can build authority and turn strangers into buyers fast, for your paid programs.
    
		One client saw a 287% Increase in buyers almost instantly.
    
		Would you like to know how?

		Template email - 

		[FIRST NAME] - Just came across your [PERSONALISATION - Name/topic of the webinar] webinar — the focus on [PERSONALISATION - Briefly mention what the webinar is all about to show that we actually saw it] sounds like a game-changer!

		This kind of value deserves a much bigger audience.
    
		Assuming we could 10X your exposure in 45 days, without you lifting a finger…
    
		We’ll do this by using the insights from the webinar and featuring them on our network of High Traffic Pages.
    
		It can build authority and turn strangers into buyers fast, for your paid programs.
    
		One client saw a 287% Increase in buyers almost instantly.
    
		Would you like to know how?
	""",
	"PODCAST": """
		Example email -

		James - Just came across your podcast about Restore More’s work with 120,000 students and 20,000 teachers – honestly, so good!

		The pod deserves a much bigger audience for so much value. 

		What if we could 10X your exposure in 45 days, no effort on your part?

		We’d pull insights from the pod & feature them on our high-traffic network.

		This builds authority and turns strangers into buyers for your paid programs fast. 

		One client saw a 287% Spike in new customers almost instantly.

		Want to hear how it works?

		Template email -
		[FIRST NAME] - Just came across your podcast about [PERSONALISATION - mention what the podcast episode was about by sharing a quick insight from it ] – honestly, so good!

		The pod deserves a much bigger audience for so much value. 

		What if we could 10X your exposure in 45 days, no effort on your part?

		We’d pull insights from the pod & feature them on our high-traffic network.

		This builds authority and turns strangers into buyers for your paid programs fast. 

		One client saw a 287% Spike in new customers almost instantly.

		Want to hear how it works?
	""",
	"EBOOK": """
		Example email - 

		James - Saw your Book on increasing sales via email list — I think it’s a great way to get people into your higher ticket stuff.

		Would be great if more people saw it.

		If we could use it to 10X your exposure in 45 days, without you lifting a finger…

		We’ll do this by using the insights from your book and feature them on our network of High Traffic Pages.

		It can build authority and turn strangers into buyers fast.

		One client saw a 287% increase in online course purchases. 

		Would you like to know how?

		Template email - 

		[First Name] - Saw your Book on [PERSONALISATION - WHAT IS THIS EBOOK ABOUT] — I think it’s a great way to get people into your higher ticket stuff.

		Would be great if more people saw it.

		If we could use it to 10X your exposure in 45 days, without you lifting a finger…

		We’ll do this by using the insights from your book and feature them on our network of High Traffic Pages.

		It can build authority and turn strangers into buyers fast.

		One client saw a 287% Increase in online course purchases. 

		Would you like to know how?
	""",
	"RECENT_BLOG": """
		Example email - 

		James - Saw your blog on Acquiring new clients with Solo Ads — really hits the mark.

		Would be great if more people read it.

		If we could use it to 10X your exposure in 45 days, without you lifting a finger…

		We’ll do this by using the insights from your blog and feature them on our network of High Traffic Pages.

		It can build authority and turn strangers into buyers fast, for your paid programs.

		One client saw a 287% Increase in new buyers almost instantly.

		Would you like to know how?

		Template email - 

		FIRST NAME - Saw your blog on [PERSONALISATION - What is the blog about ] — really hits the mark.

		Would be great if more people read it.

		If we could use it to 10X your exposure in 45 days, without you lifting a finger…

		We’ll do this by using the insights from your blog and feature them on our network of High Traffic Pages.

		It can build authority and turn strangers into buyers fast, for your paid programs.

		One client saw a 287% Increase in new buyers almost instantly.

		Would you like to know how?
	""",
	"ABOUT_US": """
		Example email - 

		James - I looked into Paramount Solutions and love your creative approach to design work.

		Feels like, a ton of people could use that.

		Assuming, we could take your existing stuff & 5X your existing exposure in 45 days…

		Without you doing any heavy lifting.

		I think there’s so much value here, the right people would straight-up flock to it.

		One client saw a 287% Increase in online course purchases. 

		Would you like to know more?

		Template email - 

		[First name] - I looked into [PERSONALISATION - Company Name] and love [PERSONALISATION - in respect of something we found interesting and insightful about their company that is clear by looking at the data point].

		Feels like, a ton of people could use that.

		Assuming, we could take your existing stuff & 5X your existing exposure in 45 days…

		Without you doing any heavy lifting.

		I think there’s so much value here, the right people would straight-up flock to it.

		One client saw a 287% Increase in online course purchases. 

		Would you like to know more?
	""",
	"NEUTRAL": """
		[FIRST NAME] – being someone with your own course/consulting offer.

		I can see how more visibility can help sell more of those.

		Assuming, we could take your existing stuff & 5X your existing exposure in 45 days…

		Without you doing any heavy lifting.

		So much value here, the right people would straight-up flock to it.

		One client saw a 287% Increase in online course purchases / consulting clients. 

		Would you like to know more?
	"""
},
"Subsequence 1": {
	"TESTIMONIALS": """
		EXAMPLE EMAIL - 

		Thanks for reaching out! The work you did with Jason and how you helped him 10x his revenue actually made me think…

		Can you help more folks make the same leap? Pretty sure more people would want that!

		Here’s a video I recorded going into detail on this: 

		Loom.com/video

		I’d love to find out more about Market Hero and how this could 10x your exposure. 

		Grab a time with me here: 

		https://calendly.com/scale-brands-lab/30min

		TEMPLATE EMAIL -

		Thanks for reaching out! The work you did with [PERSONALISATION - Mention what they did based on the testimonial/Case study/ review / results which actually brought certain results/impact based on the data point] actually made me think…

		Can you help more folks make the same leap? Pretty sure more people would want that!

		Here’s a video I recorded going into detail on this: 

		Loom.com/video

		I’d love to find out more about [PERSONALISATION - Company Name] and how this could 10x your exposure. 

		Grab a time with me here: 

		https://calendly.com/scale-brands-lab/30min
	""",
	"COURSES": """
		EXAMPLE EMAIL - 

		Thanks for reaching out! Your Copywriting 101 Course actually made me think…

		Can we use some insights from it too? Folks would be all over it!

		Here’s a video I recorded going into detail on this: 

		Loom.com/video

		I’d love to find out more about Market Hero and how this could 10x your exposure. 

		Grab a time with me here: 

		https://calendly.com/scale-brands-lab/30min

		TEMPLATE EMAIL -

		Thanks for reaching out! Your [PERSONALISATION - Mention the name of the Course/Program based on the Data Point Course/Program] (The suffix here would be based on whether it’s a course or a program based on the Data Point) actually made me think…

		Can we use some insights from it too? Folks would be all over it!

		Here’s a video I recorded going into detail on this: 

		Loom.com/video

		I’d love to find out more about [PERSONALISATION - Company Name] and how this could 10x your exposure. 

		Grab a time with me here: 

		https://calendly.com/scale-brands-lab/30min
	""",
	"SERVICES": """
		EXAMPLE EMAIL - 

		Thanks for reaching out! Your Jobsite Inspection page actually made me think…

		So many people could actually use that.

		Here’s a video I recorded going into detail on this: 

		Loom.com/video

		I’d love to find out more about Contractor Links  and how this could 10x your exposure. 

		Grab a time with me here: 

		https://calendly.com/scale-brands-lab/30min

		TEMPLATE EMAIL -

		Thanks for reaching out! Your [PERSONALISATION - Name of the Services/Consulting page based on the data point which should be relevant to the type of service they are providing to their customers] page actually made me think…

		Mind if we use some insights from it too? Folks would be all over it lol

		Here’s a video I recorded going into detail on this: 

		Loom.com/video

		I’d love to find out more about [PERSONALISATION - Their Company Name] and how this could 10x your exposure. 

		Grab a time with me here: 

		https://calendly.com/scale-brands-lab/30min
	""",
	"WEBINAR": """
		EXAMPLE EMAIL - 

		Thanks for reaching out! Your webinar on reducing client churn with email campaigns  actually made me think…

		Can we use some insights from it too? Folks would be all over it lol

		Here’s a video I recorded going into detail on this: 

		Loom.com/video

		I’d love to find out more about Market Hero and how this could 10x your exposure. 

		Grab a time with me here: 

		https://calendly.com/scale-brands-lab/30min

		TEMPLATE EMAIL -

		Thanks for reaching out! Your webinar  on [PERSONALISATION - Mention briefly about the webinar and what it’s about based on the data point] actually made me think…

		Can we use some insights from it too? Folks would be all over it lol

		Here’s a video I recorded going into detail on this: 

		Loom.com/video

		I’d love to find out more about [PERSONALISATION - Their Company Name] and how this could 10x your exposure. 

		Grab a time with me here: 

		https://calendly.com/scale-brands-lab/30min
	""",
	"PODCAST": """
		EXAMPLE EMAIL - 

		Thanks for reaching out! Your podcast about Restore More’s work with 120,000 students and 20,000 teachers actually made me think…

		Can we use some insights from it too? Folks would be all over it lol

		Here’s a video I recorded going into detail on this: 

		Loom.com/video

		I’d love to find out more about Market Hero and how this could 10x your exposure. 

		Grab a time with me here: 

		https://calendly.com/scale-brands-lab/30min

		TEMPLATE EMAIL - 

		Thanks for reaching out! Your podcast about [PERSONALISATION - Mention what the podcast what about briefly in way that makes sense to them but do it in a way that sounds like a compliment] actually made me think…

		Can we use some insights from it too? Folks would be all over it lol

		Here’s a video I recorded going into detail on this: 

		Loom.com/video

		I’d love to find out more about [PERSONALISATION - Their Company Name] and how this could 10x your exposure. 

		Grab a time with me here: 

		https://calendly.com/scale-brands-lab/30min
	""",
	"EBOOK": """
		Example Email - 

		Thanks for reaching out! Your Book on Landing High Tickets Clients via Cold Email  actually made me think…

		Can we use some insights from it too? Folks would be all over it lol

		Here’s a video I recorded going into detail on this: 

		Loom.com/video

		I’d love to find out more about Market Hero and how this could 10x your exposure. 

		Grab a time with me here: 

		https://calendly.com/scale-brands-lab/30min

		Thx, 
		Name

		Template Email - 

		Thanks for reaching out! Your Book on [PERSONALISATION - Briefly mention what their ebook is about to sum it up that actually makes sense to them based on the information gathered from the data point] actually made me think…

		Can we use some insights from it too? Folks would be all over it lol

		Here’s a video I recorded going into detail on this: 

		Loom.com/video

		I’d love to find out more about [PERSONALISATION - Name of their Company] and how this could 10x your exposure. 

		Grab a time with me here: 

		https://calendly.com/scale-brands-lab/30min
	""",
	"RECENT_BLOG": """
		EXAMPLE EMAIL -

		Thanks for reaching out! Your blog on reducing churn via retargeting email campaigns actually made me think…

		Can we use some insights from it too? Folks would be all over it lol

		Here’s a video I recorded going into detail on this: 

		Loom.com/video

		I’d love to find out more about Market Hero and how this could 10x your exposure. 

		Grab a time with me here: 

		https://calendly.com/scale-brands-lab/30min

		TEMPLATE EMAIL - 

		Thanks for reaching out! Your blog on [PERSONALISATION - What is the blog about] actually made me think…

		Can we use some insights from it too? Folks would be all over it lol

		Here’s a video I recorded going into detail on this: 

		Loom.com/video

		I’d love to find out more about [PERSONALISATION - Their Company Name] and how this could 10x your exposure. 

		Grab a time with me here: 

		https://calendly.com/scale-brands-lab/30min
	""",
	"ABOUT_US": """
		EXAMPLE EMAIL - 

		Thanks for reaching out! I looked into Market Hero and loved your approach  to creative design work.

		Feels like a ton of people could use that.

		Here’s a video I recorded going into detail on this: 

		Loom.com/video

		I’d love to find out more about Market Hero and how this could 10x your exposure. 

		Grab a time with me here: 

		https://calendly.com/scale-brands-lab/30min

		TEMPLATE EMAIL -

		Thanks for reaching out! I looked into [PERSONALISATION - Company Name] and loved [PERSONALISATION - Share what stands out/insight about their company based on the data point but use what you find as a compliment on how more people can use that as mentioned in the very next line of this template].

		Feels like a ton of people could use that.

		Here’s a video I recorded going into detail on this: 

		Loom.com/video

		I’d love to find out more about [PERSONALISATION - Company Name] and how this could 10x your exposure. 

		Grab a time with me here: 

		https://calendly.com/scale-brands-lab/30min
	""",
	"NEUTRAL": """
		Appreciate you reaching back out! I think there’s sooo much potential if we do this together.

		Here’s a quick 2 min pre-recorded video (to save myself some time LOL) we recorded going into this: 

		Loom.com/video

		I’d love to find out more about [PERSONALISATION - COMPANY NAME] and how this could 5-10x your exposure. 

		Grab a time with me here: 

		https://calendly.com/scale-brands-lab/30min
	"""
},
"Subsequence 2": """
	EXAMPLE EMAIL - 
  
	I was looking at your Website and couldn’t help but notice how you’re helping real estate folks increase their ROI while reducing costs without them actually lifting a finger.
   
	Got some time to talk about it tomorrow?

	TEMPLATE EMAIL - 
  
	I was looking at your Website and couldn’t help but notice [PERSONALISATION - specific detail, e.g., how they are helping people aka their ICP]. 
  
	Got some time to talk about it tomorrow?
""",
"Subsequence 3": """
	EXAMPLE EMAIL - 

	Knowing that AI’s blowing up in the Management Consulting Space, James.

	Like those crazy chatbots!

	I'm curious, what’s your next move to stay ahead of the curve?

	TEMPLATE EMAIL - 

	Knowing that AI’s blowing up in [PERSONALISATION - Their Industry based on the information gathered from the data points], [PERSONALISATION - First Name].

	Like those crazy chatbots!

	I'm curious, what’s your next move to stay ahead of the curve?
""",
"Subsequence 4": """
	EXAMPLE EMAIL -

	I keep wondering about how Horizon Innovations is pushing customer engagement—it’s inspiring to see your momentum!
	
	We’ve been working with some folks on similar goals, and I’d love to bounce a couple ideas your way—might be a fit. 
  
	Got a minute to talk this week?

	TEMPLATE EMAIL - 

	I keep wondering about how [PERSONALISATION - TheirCompany] is pushing [PERSONALISATION - any sort of broad topic that makes the most sense looking at the available data points].

	We’ve been working with some folks on similar goals, and I’d love to bounce a couple ideas your way—might be a fit. 

	Got a minute to talk this week?
""",
"Email 2": """
	Hate to bug you. Is this something I can pass along or no?
""",
"Email 3": """
  Hey [First Name],
  
  Just wanted to let you know…
  
  Being an Invite only firm, part of our offer is:
  
  We’ll 5x your current exposure in the next 45 days.
   
  If it doesn’t happen… then we work for FREE until we do.
  
  Would it make sense to talk about it?
"""
}

BASE_PROMPT = """
		Objective: Generate initial cold emails for outreach, following the specific template provided below. Only modify the sections within square brackets for personalisation; all other content should remain fixed.

		Instructions:
		- Personalization Fields:
			- Replace [FIRST NAME] with the name of the person given in the prompt.
			- Replace [PERSONALISATION] with a short, specific comment as per the instruction given in the square bracket of [PERSONALISATION - instruction here].
			- The short specific comment as mentioned above should provide some sort of insight and it should go along and tie in to the next line in the email template so that the overall email can make sense. 
			- While personalizing: Write the personalization in 3rd Grade level. The sentence should not be too long and complex. Use shorter sentences and simpler words.
		- Fixed Content:
			- Do not change any other text in the template. All non-bracketed content should remain exactly as written, preserving the wording, tone, and format. Be very very strict on this, I don't want anything else apart from the bracketed  content to change. 
		- Tone and Language:
			- Keep the tone friendly and professional.
			- Ensure the language is simple, conversational, and concise to stay within the ~150-word limit.
		- Don't send anything else except for the Email in the output
		- Judge if the given data point is useful for the same, if not send “NONSENSICAL DATA POINT” in the output – VERY VERY IMPORTANT 
"""

In [ ]:
def authenticate_google_sheets():
    """Authenticate and return Google Sheets service"""
    try:
        creds = service_account.Credentials.from_service_account_file(
            CREDENTIALS_FILE, 
            scopes=['https://www.googleapis.com/auth/spreadsheets']
        )
        service = build('sheets', 'v4', credentials=creds)
        return service
    except Exception as e:
        print(f"Error authenticating Google Sheets: {e}")
        return None

def initialize_openai():
    """Initialize OpenAI client"""
    try:
        client = OpenAI(api_key=OPENAI_API_KEY)
        return client
    except Exception as e:
        print(f"Error initializing OpenAI: {e}")
        return None

def count_words(text):
    """Count words in a text string"""
    if not text or text.strip() == "":
        return 0
    return len(text.strip().split())

def collect_datapoints(row):
    """Collect all datapoints from a row with their rankings"""
    datapoints = []
    
    for category, col_index in category_indices.items():
        if category == "SHOP":  # Exclude SHOP from processing
            continue
            
        if col_index < len(row):
            content = str(row[col_index]).strip()
            word_count = count_words(content)
            
            # Only include datapoints with more than 10 words
            if word_count > 10:
                datapoints.append({
                    'category': category,
                    'content': content,
                    'rank': category_ranking[category],
                    'word_count': word_count
                })
    
    # Sort by rank (ascending order - lower rank = higher priority)
    datapoints.sort(key=lambda x: x['rank'])
    return datapoints

def assign_datapoints_to_emails(datapoints):
    """Assign datapoints to emails based on ranking"""
    email_assignments = {
        'Email 1': None,
        'Subsequence 1': None,
        'Subsequence 2': None,
        'Subsequence 3': None,
        'Subsequence 4': None
    }
    
    email_order = ['Email 1', 'Subsequence 1', 'Subsequence 2', 'Subsequence 3', 'Subsequence 4']
    
    # Assign datapoints in order of their ranking
    for i, email_type in enumerate(email_order):
        if i < len(datapoints):
            email_assignments[email_type] = datapoints[i]
    
    return email_assignments

def generate_email_with_datapoint(email_type, category, datapoint_content, first_name, company_name, client):
    """Generate email using OpenAI for emails with datapoints"""
    try:
        # Handle different template structures
        if email_type in ["Email 1", "Subsequence 1"]:
            template = prompt_templates[email_type][category]
        else:
            # For Subsequence 2, 3, 4 - they have simple string templates
            template = prompt_templates[email_type]
        
        prompt = f"""
        {BASE_PROMPT}
        
        Template to use:
        {template}
        
        Person's first name: {first_name}
        Company name: {company_name}
        Datapoint content to personalize with: {datapoint_content}
        Category: {category}
        
        Generate the email following the template exactly, only replacing the bracketed placeholders.
        """
        
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=500,
            temperature=0.7
        )
        
        return response.choices[0].message.content.strip()
    
    except Exception as e:
        print(f"Error generating email with datapoint: {e}")
        return f"Error generating {email_type}"

def generate_neutral_email(email_type, first_name, company_name, website="", industry="", client=None):
    """Generate neutral email for emails without datapoints"""
    try:
        if email_type in ["Email 1", "Subsequence 1"]:
            template = prompt_templates[email_type]["NEUTRAL"]
            
            if client:
                prompt = f"""
                {BASE_PROMPT}
                
                Template to use:
                {template}
                
                Person's first name: {first_name}
                Company name: {company_name}
                
                Generate the email following the template exactly, only replacing the bracketed placeholders.
                """
                
                response = client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=500,
                    temperature=0.7
                )
                
                return response.choices[0].message.content.strip()
            else:
                # Simple replacement for neutral templates
                email = template.replace("[FIRST NAME]", first_name)
                email = email.replace("[COMPANY NAME]", company_name)
                return email.strip()
        
        elif email_type in ["Subsequence 2", "Subsequence 3", "Subsequence 4"]:
            # No neutral templates provided for these subsequences
            return "NO template provided"
        
        return f"Neutral template for {email_type}"
    
    except Exception as e:
        print(f"Error generating neutral email: {e}")
        return f"Error generating neutral {email_type}"

def generate_simple_email(email_type, first_name, company_name=""):
    """Generate simple emails for Email 2 and Email 3"""
    template = prompt_templates.get(email_type, "")
    email = template.replace("[First Name]", first_name)
    email = email.replace("[COMPANY NAME]", company_name)
    return email.strip()

def process_row(row, row_index, client):
    """Process a single row and generate all emails"""
    try:
        # Extract basic information (assuming standard positions)
        first_name = str(row[0]).strip() if len(row) > 1 else "Friend"  # Column B
        company_name = str(row[3]).strip() if len(row) > 2 else "Your Company"  # Column C
        website = str(row[6]).strip() if len(row) > 3 else ""  # Column D
        industry = str(row[8]).strip() if len(row) > 4 else ""  # Column E
        
        print(f"Processing row {row_index + 1}: {first_name} from {company_name}")
        
        # Step 1: Collect all datapoints
        datapoints = collect_datapoints(row)
        print(f"  Found {len(datapoints)} valid datapoints")
        
        # Step 2: Assign datapoints to emails
        email_assignments = assign_datapoints_to_emails(datapoints)
        
        # Step 3: Generate emails
        results = {}
        
        # Generate Email 1 and Subsequence 1
        for email_type in ['Email 1', 'Subsequence 1']:
            assignment = email_assignments[email_type]
            if assignment:
                # Generate email with datapoint
                email_content = generate_email_with_datapoint(
                    email_type, 
                    assignment['category'], 
                    assignment['content'],
                    first_name,
                    company_name,
                    client
                )
                results[email_type] = email_content
                results[f"{email_type} Data Point"] = f"{assignment['category']}: {assignment['content'][:100]}..."
            else:
                # Generate neutral email
                email_content = generate_neutral_email(
                    email_type, 
                    first_name, 
                    company_name,
                    website,
                    industry,
                    client
                )
                results[email_type] = email_content
                results[f"{email_type} Data Point"] = "No data point"
        
        # Generate Subsequence 2, 3, 4
        for email_type in ['Subsequence 2', 'Subsequence 3', 'Subsequence 4']:
            assignment = email_assignments[email_type]
            if assignment:
                # Generate email with datapoint using AI
                email_content = generate_email_with_datapoint(
                    email_type,
                    assignment['category'],
                    assignment['content'],
                    first_name,
                    company_name,
                    client
                )
                results[email_type] = email_content
                results[f"{email_type} Data Point"] = f"{assignment['category']}: {assignment['content'][:100]}..."
            else:
                # No neutral templates provided for these subsequences
                results[email_type] = "NO template provided"
                results[f"{email_type} Data Point"] = "No data point"
        
        # Generate Email 2 and Email 3
        results['Email 2'] = generate_simple_email('Email 2', first_name, company_name)
        results['Email 3'] = generate_simple_email('Email 3', first_name, company_name)
        
        return results
    
    except Exception as e:
        print(f"Error processing row {row_index + 1}: {e}")
        return None

def column_letter_to_number(letter):
    """Convert column letter to number (A=1, B=2, etc.)"""
    result = 0
    for char in letter:
        result = result * 26 + (ord(char.upper()) - ord('A') + 1)
    return result

def update_sheet_with_results(service, results_list):
    """Update the Google Sheet with generated emails"""
    try:
        # Prepare batch update data
        data = []
        
        for row_index, results in enumerate(results_list):
            if results is None:
                continue
                
            actual_row = row_index + 2  # Assuming header is row 1, data starts from row 2
            
            for email_type, column_letter in output_columns.items():
                if email_type in results:
                    cell_range = f"{SHEET_NAME}!{column_letter}{actual_row}"
                    data.append({
                        'range': cell_range,
                        'values': [[results[email_type]]]
                    })
        
        if data:
            body = {
                'valueInputOption': 'RAW',
                'data': data
            }
            
            result = service.spreadsheets().values().batchUpdate(
                spreadsheetId=SPREADSHEET_ID,
                body=body
            ).execute()
            
            print(f"Updated {len(data)} cells in the spreadsheet")
            return True
        else:
            print("No data to update")
            return False
    
    except Exception as e:
        print(f"Error updating sheet: {e}")
        return False
    
def update_sheet_with_batch_results(service, batch_results):
    """Update the Google Sheet with results from a batch of rows"""
    try:
        # Prepare batch update data
        data = []
        for row_index, results in batch_results:
            if results is None:
                continue
                
            actual_row = row_index + 2  # Assuming header is row 1, data starts from row 2
            for email_type, column_letter in output_columns.items():
                if email_type in results:
                    cell_range = f"{SHEET_NAME}!{column_letter}{actual_row}"
                    data.append({
                        'range': cell_range,
                        'values': [[results[email_type]]]
                    })
        if data:
            body = {
                'valueInputOption': 'RAW',
                'data': data
            }
            result = service.spreadsheets().values().batchUpdate(
                spreadsheetId=SPREADSHEET_ID,
                body=body
            ).execute()
            print(f"  📝 Updated {len(data)} cells in the spreadsheet")
            return True
        else:
            print("  ⚠️  No data to update in this batch")
            return False
    except Exception as e:
        print(f"  ❌ Error updating batch: {e}")
        return False


def main(start_row=1):
    """Main function to run the email generation process with batch processing
    Args:
        start_row (int): Row number to start processing from (1-based, excluding header)
    """
    print(f"Starting email generation process from row {start_row}...")
    # Initialize services
    sheets_service = authenticate_google_sheets()
    if not sheets_service:
        print("Failed to authenticate Google Sheets")
        return
    openai_client = initialize_openai()
    if not openai_client:
        print("Failed to initialize OpenAI client")
        return
    try:
        # Read data from Google Sheets
        range_name = f"{SHEET_NAME}!A:U"  # Read all data up to column U
        result = sheets_service.spreadsheets().values().get(
            spreadsheetId=SPREADSHEET_ID,
            range=range_name
        ).execute()
        values = result.get('values', [])
        if not values:
            print('No data found in the sheet.')
            return
        # Skip header row and validate start_row
        data_rows = values[1:]
        total_rows = len(data_rows)
        print(f"Found {total_rows} data rows in the sheet")
        # Validate start_row
        if start_row < 1:
            print("Error: start_row must be >= 1")
            return
        elif start_row > total_rows:
            print(f"Error: start_row ({start_row}) is greater than total rows ({total_rows})")
            return
        # Adjust for 0-based indexing (start_row is 1-based)
        start_index = start_row - 1
        remaining_rows = data_rows[start_index:]
        remaining_count = len(remaining_rows)
        print(f"Starting from row {start_row}, processing {remaining_count} remaining rows")
        batch_size = 10
        successful_batches = 0
        # Process rows in batches of 10, starting from start_index
        for batch_start in range(0, remaining_count, batch_size):
            batch_end = min(batch_start + batch_size, total_rows)
            batch_rows = data_rows[batch_start:batch_end]
            print(f"\nProcessing batch {successful_batches + 1}: rows {batch_start + 1} to {batch_end}")
            # Process each row in the current batch
            batch_results = []
            for i, row in enumerate(batch_rows):
                actual_row_index = batch_start + i  # Global row index
                results = process_row(row, actual_row_index, openai_client)
                batch_results.append((actual_row_index, results))  # Store with row index
                # Optional: Add delay to avoid rate limiting
                import time
                time.sleep(1)
            # Update the sheet with this batch's results
            if update_sheet_with_batch_results(sheets_service, batch_results):
                successful_batches += 1
                print(f"✅ Batch {successful_batches} completed successfully!")
            else:
                print(f"❌ Failed to update batch {successful_batches + 1}")
                print("Stopping process to prevent data loss...")
                break
        print(f"\n🎉 Email generation completed! Successfully processed {successful_batches} batches.")
    except Exception as e:
        print(f"Error in main process: {e}")
        
if __name__ == "__main__":
    main()